# Notebook Huấn luyện & Đánh giá Mô hình Phân loại (Classification Notebook)

Notebook này xây dựng, huấn luyện và đánh giá các thuật toán học máy phân loại trễ chuyến bay:
1. **Mô hình 1: Phân loại Khởi hành trễ** (`TARGET_DEP_DELAY = 1[DEP_DELAY >= 15]`)
2. **Mô hình 2: Phân loại Đến trễ** (`TARGET_ARR_DELAY = 1[ARR_DELAY >= 15]`)

Các thuật toán so sánh gồm: **XGBoost (GPU/Hist)**, **Logistic Regression**, và **Random Forest**.

### Cell 1: Import các thư viện cần thiết

In [1]:
import os
import sys
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score

# Cấu hình cảnh báo và hiển thị
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
print('Tất cả các thư viện đã được import thành công!')

Tất cả các thư viện đã được import thành công!


### Cell 2: Tải và nạp dữ liệu chuyến bay (2016-2024)
Tự động tìm kiếm đường dẫn dữ liệu tối ưu từ thư mục gốc hoặc thư mục hiện tại của notebook và nạp tập dữ liệu.

In [2]:
candidate_paths = [
    'data/processed/inbound_atl',
    '../../data/processed/inbound_atl',
    '../data/processed/inbound_atl',
    'data/processed/tabular_by_year',
    '../../data/processed/tabular_by_year',
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError('Không tìm thấy thư mục dữ liệu đã xử lý (inbound_atl hoặc tabular_by_year)!')

print(f'Đang tải dữ liệu từ: {data_path} ...')
df = pd.read_parquet(data_path)

# Tiền xử lý các cột thời gian thành giờ
df['CRS_DEP_HOUR'] = pd.to_datetime(df['CRS_DEP_TIME'], errors='coerce').dt.hour
df['CRS_ARR_HOUR'] = pd.to_datetime(df['CRS_ARR_TIME'], errors='coerce').dt.hour

# Xử lý missing values thời tiết bằng median
weather_cols = ['O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD']
imputer = SimpleImputer(strategy='median')
df[weather_cols] = imputer.fit_transform(df[weather_cols])

# Mã hóa LabelEncoder cho các cột Categorical
for col in ['OP_CARRIER', 'ORIGIN', 'DEST']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Loại bỏ các hàng thiếu các giá trị cốt lõi
df = df.dropna(subset=['CRS_ELAPSED_TIME', 'CRS_DEP_HOUR', 'CRS_ARR_HOUR', 'ARR_DELAY', 'DEP_DELAY'])

print(f'Kích thước dữ liệu sẵn sàng cho huấn luyện: {df.shape}')
print('Phân phối số lượng chuyến bay theo năm:')
print(df['source_year'].value_counts().sort_index())

Đang tải dữ liệu từ: ../../data/processed/inbound_atl ...


Kích thước dữ liệu sẵn sàng cho huấn luyện: (3022433, 40)
Phân phối số lượng chuyến bay theo năm:
source_year
2016    381166
2017    358263
2018    386580
2019    391075
2020    242121
2021    309621
2022    311701
2023    332741
2024    309165
Name: count, dtype: int64


### Cell 3: Hàm chuẩn bị dữ liệu và huấn luyện mô hình (prepare_and_train)
Hàm chuẩn hóa và huấn luyện mô hình với cấu hình đặc trưng tùy biến theo từng bài toán trễ, phân chia Temporal Split chuẩn (Train 2016-2022, Valid 2023, Test 2024), lưu trữ các chỉ số để tổng hợp so sánh.

In [3]:
# Danh sách lưu trữ kết quả đánh giá để so sánh tổng hợp
all_evaluation_metrics = []

def prepare_and_train(df_full, target_col, exclude_cols, drop_weather_dest=False, model_types=['xgb', 'lr', 'rf']):
    print(f"{'='*50}")
    print(f"BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: {target_col} >= 15 phút")
    print(f"CÁC MÔ HÌNH ĐƯỢC CHỌN: {model_types}")
    print(f"{'='*50}")
    
    df_model = df_full.copy()
    
    target_name = f'TARGET_{target_col}'
    df_model[target_name] = (df_model[target_col] >= 15).astype(int)
    
    cols_to_drop = exclude_cols + ['FL_DATE', 'flight_key', 'OP_CARRIER_FL_NUM', 'source_row_number', 'FLIGHTS', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'ARR_DELAY', 'DEP_DELAY']
    
    if drop_weather_dest:
        print("-> Đang loại bỏ các trường thời tiết ở nơi đến (D_TEMP, D_PRCP, D_WSPD)...")
        cols_to_drop += ['D_TEMP', 'D_PRCP', 'D_WSPD', 'D_LATITUDE', 'D_LONGITUDE', 'DEST_INDEX']
        
    df_model = df_model.drop(columns=cols_to_drop, errors='ignore')
    
    train_mask = df_model['source_year'].between(2016, 2022)
    valid_mask = df_model['source_year'] == 2023
    test_mask = df_model['source_year'] == 2024
    
    X_train = df_model[train_mask].drop(columns=[target_name, 'source_year']).copy()
    y_train = df_model[train_mask][target_name].copy()
    
    X_valid = df_model[valid_mask].drop(columns=[target_name, 'source_year']).copy()
    y_valid = df_model[valid_mask][target_name].copy()
    
    X_test = df_model[test_mask].drop(columns=[target_name, 'source_year']).copy()
    y_test = df_model[test_mask][target_name].copy()
    
    print(f"Kích thước Train (2016-2022): {X_train.shape}")
    print(f"Kích thước Valid (2023)    : {X_valid.shape}")
    print(f"Kích thước Test (2024)     : {X_test.shape}")
    
    del df_model
    gc.collect()
    
    numeric_cols = [col for col in X_train.columns if col not in ['OP_CARRIER', 'ORIGIN', 'DEST', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK']]
    scaler = StandardScaler()
    
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_valid[numeric_cols] = scaler.transform(X_valid[numeric_cols])
    X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])
    
    # Chuẩn hóa kiểu dữ liệu cho mô hình
    for c in X_train.columns:
        if str(X_train[c].dtype).startswith('Int') or str(X_train[c].dtype).startswith('int'):
            X_train[c] = X_train[c].astype(np.int32)
            X_valid[c] = X_valid[c].astype(np.int32)
            X_test[c] = X_test[c].astype(np.int32)
        elif str(X_train[c].dtype).startswith('Float') or str(X_train[c].dtype).startswith('float'):
            X_train[c] = X_train[c].astype(np.float32)
            X_valid[c] = X_valid[c].astype(np.float32)
            X_test[c] = X_test[c].astype(np.float32)
            
    trained_models = {}
    
    for m_type in model_types:
        print(f"{'-'*40}")
        print(f"Đang huấn luyện mô hình {m_type.upper()}...")
        if m_type == 'rf':
            clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
        elif m_type == 'xgb':
            try:
                import xgboost as xgb
                try:
                    clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cuda')
                    clf.fit(X_train.iloc[:5], y_train.iloc[:5])
                except Exception:
                    print("-> GPU không khả dụng cho XGBoost, chuyển sang sử dụng CPU...")
                    clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cpu', n_jobs=-1)
                clf = xgb.XGBClassifier(n_estimators=100, max_depth=10, random_state=42, tree_method='hist', device='cuda')
            except ImportError:
                from sklearn.ensemble import HistGradientBoostingClassifier
                print("-> Không tìm thấy XGBoost, sử dụng HistGradientBoostingClassifier...")
                clf = HistGradientBoostingClassifier(max_iter=100, max_depth=10, random_state=42)
        elif m_type == 'lr':
            try:
                from cuml.linear_model import LogisticRegression as cuLogisticRegression
                print("Sử dụng cuML Logistic Regression (GPU)...")
                clf = cuLogisticRegression(max_iter=200)
            except (ImportError, Exception):
                from sklearn.linear_model import LogisticRegression
                print("Không tìm thấy cuML, fallback về sklearn Logistic Regression (CPU)...")
                clf = LogisticRegression(max_iter=200)
        else:
            print(f"Bỏ qua {m_type}: model_type không hợp lệ.")
            continue
        
        clf.fit(X_train, y_train)
        
        # Đánh giá trên tập Validation
        y_val_pred = clf.predict(X_valid)
        y_val_prob = clf.predict_proba(X_valid)[:, 1]
        val_auc = roc_auc_score(y_valid, y_val_prob)
        val_acc = accuracy_score(y_valid, y_val_pred)
        print(f"Validation ROC-AUC: {val_auc:.4f}")
        
        # Đánh giá trên tập Test
        y_test_pred = clf.predict(X_test)
        y_test_prob = clf.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_test_prob)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        print(f"--- Báo cáo kết quả phân loại {m_type.upper()} (Test 2024) ---")
        print(classification_report(y_test, y_test_pred))
        print(f"Test ROC-AUC Score: {test_auc:.4f}")
        
        all_evaluation_metrics.append({
            'Task': f'{target_col} >= 15',
            'Model': m_type.upper(),
            'Val ROC-AUC': round(val_auc, 4),
            'Test ROC-AUC': round(test_auc, 4),
            'Test Accuracy': round(test_acc, 4),
            'Test F1-Score': round(test_f1, 4)
        })
        
        trained_models[m_type] = clf
        
    return trained_models

### Cell 4: Mô hình 1 - Dự đoán Khởi hành trễ (DEP_DELAY >= 15)
Đối với dự đoán khởi hành trễ, loại bỏ các thông tin rò rỉ diễn ra sau khi máy bay bắt đầu lăn bánh hoặc cất cánh.

In [4]:
# Các cột không được sử dụng khi dự đoán Khởi hành trễ
leakage_dep = [
    'DEP_TIME', 'TAXI_OUT', 'WHEELS_OFF', 
    'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME'
]

models_dep = prepare_and_train(
    df, 
    target_col='DEP_DELAY', 
    exclude_cols=leakage_dep, 
    drop_weather_dest=False,
    model_types=['xgb', 'lr', 'rf']
)

BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: DEP_DELAY >= 15 phút
CÁC MÔ HÌNH ĐƯỢC CHỌN: ['xgb', 'lr', 'rf']


Kích thước Train (2016-2022): (2380527, 22)
Kích thước Valid (2023)    : (332741, 22)
Kích thước Test (2024)     : (309165, 22)


----------------------------------------
Đang huấn luyện mô hình XGB...


Validation ROC-AUC: 0.6916


--- Báo cáo kết quả phân loại XGB (Test 2024) ---
              precision    recall  f1-score   support

           0       0.84      0.97      0.90    254658
           1       0.45      0.11      0.17     54507

    accuracy                           0.82    309165
   macro avg       0.64      0.54      0.54    309165
weighted avg       0.77      0.82      0.77    309165

Test ROC-AUC Score: 0.6837
----------------------------------------
Đang huấn luyện mô hình LR...
Không tìm thấy cuML, fallback về sklearn Logistic Regression (CPU)...


Validation ROC-AUC: 0.6651
--- Báo cáo kết quả phân loại LR (Test 2024) ---


              precision    recall  f1-score   support

           0       0.82      1.00      0.90    254658
           1       0.58      0.00      0.00     54507

    accuracy                           0.82    309165
   macro avg       0.70      0.50      0.45    309165
weighted avg       0.78      0.82      0.74    309165

Test ROC-AUC Score: 0.6610
----------------------------------------
Đang huấn luyện mô hình RF...


Validation ROC-AUC: 0.6932


--- Báo cáo kết quả phân loại RF (Test 2024) ---
              precision    recall  f1-score   support

           0       0.82      1.00      0.90    254658
           1       0.58      0.00      0.01     54507

    accuracy                           0.82    309165
   macro avg       0.70      0.50      0.46    309165
weighted avg       0.78      0.82      0.75    309165

Test ROC-AUC Score: 0.6893


### Cell 5: Mô hình 2 - Dự đoán Đến trễ (ARR_DELAY >= 15)
Dự đoán trước khi khởi hành: Loại bỏ các biến diễn ra trong và sau chuyến bay, đồng thời loại bỏ các trường thời tiết nơi đến (`D_*`) theo hợp đồng nghiên cứu.

In [5]:
# Các cột rò rỉ dữ liệu cho Arrival Delay
leakage_arr = [
    'DEP_TIME', 'TAXI_OUT', 'WHEELS_OFF', 
    'WHEELS_ON', 'TAXI_IN', 'ARR_TIME', 
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME'
]

models_arr = prepare_and_train(
    df, 
    target_col='ARR_DELAY', 
    exclude_cols=leakage_arr, 
    drop_weather_dest=True,  # Loại bỏ D_TEMP, D_PRCP, D_WSPD theo yêu cầu
    model_types=['xgb', 'lr', 'rf']
)

BẮT ĐẦU HUẤN LUYỆN CÁC MÔ HÌNH DỰ ĐOÁN: ARR_DELAY >= 15 phút
CÁC MÔ HÌNH ĐƯỢC CHỌN: ['xgb', 'lr', 'rf']


-> Đang loại bỏ các trường thời tiết ở nơi đến (D_TEMP, D_PRCP, D_WSPD)...


Kích thước Train (2016-2022): (2380527, 16)
Kích thước Valid (2023)    : (332741, 16)
Kích thước Test (2024)     : (309165, 16)


----------------------------------------
Đang huấn luyện mô hình XGB...


Validation ROC-AUC: 0.6822


--- Báo cáo kết quả phân loại XGB (Test 2024) ---
              precision    recall  f1-score   support

           0       0.83      0.98      0.90    255127
           1       0.44      0.09      0.15     54038

    accuracy                           0.82    309165
   macro avg       0.64      0.53      0.52    309165
weighted avg       0.77      0.82      0.77    309165

Test ROC-AUC Score: 0.6690
----------------------------------------
Đang huấn luyện mô hình LR...
Không tìm thấy cuML, fallback về sklearn Logistic Regression (CPU)...


Validation ROC-AUC: 0.6416
--- Báo cáo kết quả phân loại LR (Test 2024) ---


              precision    recall  f1-score   support

           0       0.83      1.00      0.90    255127
           1       0.00      0.00      0.00     54038

    accuracy                           0.83    309165
   macro avg       0.41      0.50      0.45    309165
weighted avg       0.68      0.83      0.75    309165

Test ROC-AUC Score: 0.6325
----------------------------------------
Đang huấn luyện mô hình RF...


Validation ROC-AUC: 0.6900


--- Báo cáo kết quả phân loại RF (Test 2024) ---
              precision    recall  f1-score   support

           0       0.83      1.00      0.90    255127
           1       0.63      0.00      0.01     54038

    accuracy                           0.83    309165
   macro avg       0.73      0.50      0.46    309165
weighted avg       0.79      0.83      0.75    309165

Test ROC-AUC Score: 0.6826


### Cell 6: Bảng so sánh tổng hợp hiệu năng các mô hình
Hiển thị bảng tổng kết ROC-AUC, Độ chính xác (Accuracy), và F1-Score của cả hai bài toán dự đoán trên các tập Validation (2023) và Test (2024).

In [6]:
df_summary = pd.DataFrame(all_evaluation_metrics)
print("="*70)
print("TỔNG HỢP SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH PHÂN LOẠI")
print("="*70)
print(df_summary.to_string(index=False))

TỔNG HỢP SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH PHÂN LOẠI
           Task Model  Val ROC-AUC  Test ROC-AUC  Test Accuracy  Test F1-Score
DEP_DELAY >= 15   XGB       0.6916        0.6837         0.8195         0.1722
DEP_DELAY >= 15    LR       0.6651        0.6610         0.8238         0.0028
DEP_DELAY >= 15    RF       0.6932        0.6893         0.8239         0.0089
ARR_DELAY >= 15   XGB       0.6822        0.6690         0.8212         0.1459
ARR_DELAY >= 15    LR       0.6416        0.6325         0.8252         0.0000
ARR_DELAY >= 15    RF       0.6900        0.6826         0.8254         0.0058
